# Frozen probe controls, explained visually

## tl;dr

This notebook asks a simple question: **what image information changes the predicted matter-density parameter, $\Omega_m$?** It shows the actual 2D map before and after each controlled change, then places the saved $\Omega_m$ prediction beside each image.

The prediction model is **not trained again** during these tests. The VGG image encoder and MLP prediction head were trained once and frozen. Every panel follows the same path:

`same map -> one controlled change -> SAME frozen predictor -> compare Omega_m prediction`

**No refitting. No control-specific calibration. No new neural-network inference in this notebook.** The displayed predictions are read from the completed control CSV files.

The default Great Lakes paths point to the result checkout at `/home/jiamingp/diffusion_models_repo` and the pinned control-code checkout at `/scratch/huterer_root/huterer0/jiamingp/probe_controls_code_456f01a`.

## Context & Methods

The same frozen predictor estimates six simulation parameters from every image. The main walkthrough focuses on $\Omega_m$ so that the visual comparison stays readable. Results for the other five parameters appear later under **Technical details**.

- **C0: orientation and location.** Rotate, reflect, or periodically shift the same map. Its physical content is unchanged.
- **C1: structure size.** Keep only broad structures (low-pass) or only fine structures (high-pass).
- **C4: missing-power explanation.** Alter real maps to imitate the generated maps' measured loss of spatial power, then compare with generated maps.

### Key Assumptions

- The source checkout and saved result manifests are frozen at the expected commit.
- Held-out simulations are exactly 900 through 931.
- The example is selected deterministically from true $\Omega_m$ only; prediction error is never used for selection.
- Every image label comes from a saved prediction row for the exact simulation, slice/sample, transform, source, and run.
- In C4, the real and generated examples use the same cosmology but are different realizations. They are not pixel-paired.
- Matching two-point power does not match the one-point distribution or higher-order structure.

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except (NameError, AttributeError):
    pass
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

class MissingInputError(FileNotFoundError):
    pass

PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', '/home/jiamingp/diffusion_models_repo')).expanduser().resolve()
RESULTS_ROOT = Path(os.environ.get('PROBE_CONTROLS_RESULTS_ROOT', str(PROJECT_DIR / 'results' / 'nf_conditional_bias_probe'))).expanduser().resolve()
CODE_ROOT = Path(os.environ.get('PROBE_CONTROLS_CODE_ROOT', '/scratch/huterer_root/huterer0/jiamingp/probe_controls_code_456f01a')).expanduser().resolve()
EXPECTED_COMMIT = 'dced4f8928efe248d819a72560ef61a099d0c4a3'
EXPECTED_RUNS = {'nf_cond_bias_hi_u128_d2p07_n128_200k', 'nf_cond_bias_hi_u128_d2p14_n16384_200k'}
EXPECTED_HELDOUT = np.arange(900, 932, dtype=np.int64)
PARAMETER_ORDER = ('Omega_m', 'sigma_8', 'A_SN1', 'A_AGN1', 'A_SN2', 'A_AGN2')
PARAMETERS = set(PARAMETER_ORDER)
FOCUS_PARAMETER = os.environ.get('PROBE_CONTROLS_PARAMETER', 'Omega_m')
if FOCUS_PARAMETER not in PARAMETERS:
    raise ValueError(f'PROBE_CONTROLS_PARAMETER must be one of {sorted(PARAMETERS)}; got {FOCUS_PARAMETER!r}')
C4_LIMITATION = 'The transfer controls match only the measured two-point power deficit. They do not match the one-point PDF or higher-order structure, so a negative result rules out only that two-point deficit as the explanation.'

TRANSFORM_DIR = RESULTS_ROOT / 'transform_controls'
C4_DIR = RESULTS_ROOT / 'degradation_controls'
MANIFEST_PATH = PROJECT_DIR / 'local' / 'nf_conditional_bias_probe' / 'manifest.json'
ENCODER_PATH = RESULTS_ROOT / 'encoder' / 'vgg_mlp_encoder.npz'
HEAD_PATH = RESULTS_ROOT / 'encoder' / 'vgg_mlp_encoder.pkl'
VGG_WEIGHTS_PATH = Path(os.environ.get('PROBE_CONTROLS_VGG_WEIGHTS', '/scratch/huterer_root/huterer0/jiamingp/torch_cache/hub/checkpoints/vgg16-397923af.pth')).expanduser().resolve()
TRANSFORM_CSV = TRANSFORM_DIR / 'probe_transform_predictions.csv'
C4_CSV = C4_DIR / 'probe_degradation_predictions.csv'
DATA_ROOT = Path(os.environ.get('PROBE_CONTROLS_DATA_ROOT', '/scratch/huterer_root/huterer0/CAMELS/CMD/3d_grids/IllustrisTNG')).expanduser().resolve()
C0_GALLERY_COLUMNS = ('original', 'rotated', 'reflected', 'shifted')
C1_GALLERY_COLUMNS = ('original', 'low-pass', 'high-pass')
C4_GALLERY_COLUMNS = ('original real', 'power-matched real', 'Gaussian-smoothed real', 'generated')
RUN_LABELS = {'nf_cond_bias_hi_u128_d2p07_n128_200k': 'N=128, d=2.07', 'nf_cond_bias_hi_u128_d2p14_n16384_200k': 'N=16,384, d=2.14'}

print('PROJECT_DIR =', PROJECT_DIR)
print('RESULTS_ROOT =', RESULTS_ROOT)
print('CODE_ROOT =', CODE_ROOT)
print('EXPECTED_COMMIT =', EXPECTED_COMMIT)
print('DATA_ROOT =', DATA_ROOT)

## Data

The next two cells check that the code version, result manifests, held-out simulations, schemas, and row counts are exactly the ones used for the completed controls. They deliberately fail instead of drawing a figure from incomplete or mismatched files. The source check is equivalent to `git rev-parse HEAD`, and the generated-run manifest is `local/nf_conditional_bias_probe/manifest.json`.

The large prediction table is read in chunks. Later, the visual walkthrough loads only the rows and 2D examples required for the displayed panels.

Optional unattended execution on Great Lakes:

```bash
/home/jiamingp/venvs/cosmodiff_nf_class/bin/python -m jupyter nbconvert --execute --to notebook \
  --output-dir /home/jiamingp/diffusion_models_repo/notebooks/executed \
  --output nf_probe_control_validation.executed.ipynb \
  /home/jiamingp/diffusion_models_repo/notebooks/nf_probe_control_validation.ipynb
```

In [ ]:
def require_file(path):
    path = Path(path)
    if not path.is_file():
        raise MissingInputError(f'Missing required input: {path}')
    return path

def git_head(root):
    metadata = Path(root) / '.git'
    if metadata.is_file():
        pointer = metadata.read_text().strip().split(':', 1)[-1].strip()
        metadata = (metadata.parent / pointer).resolve()
    head = require_file(metadata / 'HEAD').read_text().strip()
    if head.startswith('ref:'):
        return require_file(metadata / head.split(':', 1)[1].strip()).read_text().strip()
    return head

if git_head(CODE_ROOT) != EXPECTED_COMMIT:
    raise RuntimeError(f'Expected CODE_ROOT commit {EXPECTED_COMMIT}; found {git_head(CODE_ROOT)}')

required_inputs = [
    ENCODER_PATH, HEAD_PATH, MANIFEST_PATH, VGG_WEIGHTS_PATH,
    TRANSFORM_CSV, C4_CSV,
    TRANSFORM_DIR / 'probe_transform_metrics.json',
    TRANSFORM_DIR / 'c0_symmetry_summary.json',
    TRANSFORM_DIR / 'c1_scale_cut_summary.json',
    TRANSFORM_DIR / 'manifest.json',
    C4_DIR / 'probe_degradation_metrics.json',
    C4_DIR / 'power_transfer_curves.json',
    C4_DIR / 'field_histograms.json',
    C4_DIR / 'manifest.json',
]
for input_path in required_inputs:
    require_file(input_path)

def strict_json(path):
    def reject_constant(value):
        raise ValueError(f'Non-strict JSON constant {value} in {path}')
    def reject_duplicate_keys(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f'Duplicate JSON key {key!r} in {path}')
            result[key] = value
        return result
    return json.loads(Path(path).read_text(), parse_constant=reject_constant, object_pairs_hook=reject_duplicate_keys)

def assert_clean_manifest(payload, path):
    if not isinstance(payload, dict) or not isinstance(payload.get('git'), dict):
        raise ValueError(f'Manifest lacks Git provenance: {path}')
    git = payload['git']
    if git.get('revision') != EXPECTED_COMMIT or git.get('dirty') is not False:
        raise ValueError(f'Manifest is not clean at the expected commit: {path}')

transform_manifest = strict_json(TRANSFORM_DIR / 'manifest.json')
c4_manifest = strict_json(C4_DIR / 'manifest.json')
assert_clean_manifest(transform_manifest, TRANSFORM_DIR / 'manifest.json')
assert_clean_manifest(c4_manifest, C4_DIR / 'manifest.json')

with np.load(ENCODER_PATH, allow_pickle=True) as encoder_data:
    heldout = np.asarray(encoder_data['heldout_indices'], dtype=np.int64)
if not np.array_equal(heldout, EXPECTED_HELDOUT):
    raise ValueError(f'heldout_indices must be 900..931; found {heldout.tolist()}')

manifest_rows = strict_json(MANIFEST_PATH)
if not isinstance(manifest_rows, list):
    raise ValueError('The generated-run manifest must be a JSON list')
manifest_runs = {str(row.get('run_name')) for row in manifest_rows}
if not EXPECTED_RUNS.issubset(manifest_runs):
    raise ValueError(f'Manifest is missing expected runs: {sorted(EXPECTED_RUNS - manifest_runs)}')
print('Structural input gates passed for', len(manifest_rows), 'manifest rows')

In [ ]:
TRANSFORM_REQUIRED = {
    'transform', 'transform_family', 'k_cut', 'k_cut_over_knyq', 'window',
    'dihedral_g', 'roll_dx', 'roll_dy', 'sim_index', 'z_index',
    'parameter', 'theta_true', 'theta_pred', 'out_of_range_fraction',
}
C4_REQUIRED = TRANSFORM_REQUIRED | {'source', 'run_name', 'dataset_size'}
EXPECTED_TRANSFORM_LINES = 1_990_657
EXPECTED_C4_LINES = 73_729

def validate_prediction_csv(path, expected_lines, required_columns, key_columns, require_all_simulations=True):
    header = pd.read_csv(path, nrows=0)
    missing = sorted(set(required_columns) - set(header.columns))
    if missing:
        raise ValueError(f'{path} is missing required columns: {missing}')
    with Path(path).open('rb') as stream:
        line_count = sum(1 for _ in stream)
    if line_count != expected_lines:
        raise ValueError(f'{path} has {line_count} lines; expected {expected_lines}')
    seen_hashes = set()
    transforms, transform_families, parameters, simulations, run_names = set(), set(), set(), set(), set()
    rows = 0
    usecols = sorted(set(required_columns) | set(key_columns))
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=200_000):
        rows += len(chunk)
        transforms.update(chunk['transform'].dropna().astype(str))
        transform_families.update(chunk['transform_family'].dropna().astype(str))
        parameters.update(chunk['parameter'].dropna().astype(str))
        simulations.update(chunk['sim_index'].dropna().astype(int))
        if 'run_name' in chunk:
            run_names.update(chunk['run_name'].dropna().astype(str))
        keys = chunk[key_columns].astype(object).where(chunk[key_columns].notna(), '<NA>')
        hashes = pd.util.hash_pandas_object(keys, index=False).astype('uint64')
        if hashes.duplicated().any() or set(hashes.tolist()) & seen_hashes:
            raise ValueError(f'{path} contains duplicate analytical grains')
        seen_hashes.update(hashes.tolist())
    if rows + 1 != line_count:
        raise ValueError(f'{path} row/line reconciliation failed')
    if (require_all_simulations and simulations != set(EXPECTED_HELDOUT)) or (not require_all_simulations and not simulations.issubset(set(EXPECTED_HELDOUT))) or parameters != PARAMETERS:
        raise ValueError(f'{path} does not cover heldout simulations and all six parameters')
    return {'transforms': transforms, 'transform_families': transform_families, 'parameters': parameters, 'simulations': simulations, 'run_names': run_names}

transform_coverage = validate_prediction_csv(
    TRANSFORM_CSV, EXPECTED_TRANSFORM_LINES, TRANSFORM_REQUIRED,
    ['transform', 'transform_family', 'sim_index', 'z_index', 'parameter'],
)
c4_coverage = validate_prediction_csv(
    C4_CSV, EXPECTED_C4_LINES, C4_REQUIRED,
    ['transform', 'transform_family', 'source', 'run_name', 'dataset_size', 'sim_index', 'z_index', 'parameter'],
    require_all_simulations=False,
)
transform_names = transform_coverage['transforms']
transform_families = transform_coverage['transform_families']
c4_runs = c4_coverage['run_names']
if 'identity' not in transform_names:
    raise ValueError('C0 identity coverage is incomplete')
if not {'lowpass', 'highpass', 'fft_roundtrip_null'}.issubset(transform_families):
    raise ValueError(f'C1 transform-family coverage is incomplete: {sorted(transform_families)}')
if not EXPECTED_RUNS.issubset(c4_runs):
    raise ValueError('C4 run coverage is incomplete')
print('CSV gates passed without loading either large prediction file fully')

## Results

### First: what stays fixed?

Only the image changes. The trained predictor does not:

`same map -> one controlled change -> SAME frozen predictor -> compare Omega_m prediction`

**No refitting occurs.** The images below are reconstructed with the recorded transformations, while every numerical prediction is looked up from the saved CSV output of the completed run.

The example-selection rule is fixed before looking at predictions: choose the evaluation cosmology in the middle of the ordered true-$\Omega_m$ values, then choose its smallest available slice/sample index. This prevents cherry-picking a visually convenient success or failure.

In [ ]:
def select_representative_example(rows):
    frame = rows.copy()
    if 'parameter' in frame:
        frame = frame[frame['parameter'].astype(str) == 'Omega_m'].copy()
    assert len(frame) > 0, 'No Omega_m rows are available for deterministic selection'
    cosmologies = (
        frame[['sim_index', 'theta_true']]
        .drop_duplicates()
        .sort_values(['theta_true', 'sim_index'])
        .reset_index(drop=True)
    )
    chosen = cosmologies.iloc[len(cosmologies) // 2]
    sim_index = int(chosen['sim_index'])
    z_index = int(frame.loc[frame['sim_index'].astype(int) == sim_index, 'z_index'].astype(int).min())
    return {'sim_index': sim_index, 'z_index': z_index, 'theta_true': float(chosen['theta_true'])}

In [ ]:
import re
import sys

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from simdiff_eval.probe_eval import load_heldout_real_slices
from simdiff_eval.probe_transforms import (
    gaussian_smoothing_transform,
    get_transform,
    transfer_transform,
)

def read_saved_prediction_rows(path, filters, chunksize=200_000):
    selected = []
    for chunk in pd.read_csv(path, chunksize=chunksize):
        keep = np.ones(len(chunk), dtype=bool)
        for column, wanted in filters.items():
            values = wanted if isinstance(wanted, (list, tuple, set, np.ndarray)) else [wanted]
            keep &= chunk[column].isin(list(values)).to_numpy()
        if keep.any():
            selected.append(chunk.loc[keep].copy())
    if not selected:
        raise ValueError(f'No saved prediction rows matched {filters}')
    return pd.concat(selected, ignore_index=True)

def one_saved_prediction(rows, **keys):
    selected = rows.copy()
    for column, value in keys.items():
        selected = selected[selected[column] == value]
    if len(selected) != 1:
        raise ValueError(f'Expected one saved prediction row for {keys}; found {len(selected)}')
    return selected.iloc[0]

def saved_transform_records(rows, transform_names):
    records = {name: one_saved_prediction(rows, transform=name) for name in transform_names}
    truth_values = np.asarray([float(record['theta_true']) for record in records.values()])
    if not np.allclose(truth_values, truth_values[0], rtol=0.0, atol=1.0e-7):
        raise ValueError(f'Saved transform rows disagree on truth values: {truth_values.tolist()}')
    return records

def c4_saved_records_for_display(rows, *, run_name, dataset_size, real_z_index, generated_sample_index):
    suffix = f'{run_name}__N{int(dataset_size)}'
    records = {
        'original real': one_saved_prediction(rows, source='real_original', run_name='real_original', transform='identity', z_index=int(real_z_index)),
        'power-matched real': one_saved_prediction(rows, source='real_measured_transfer', run_name=run_name, dataset_size=int(dataset_size), transform=f'transfer_Tk__{suffix}', z_index=int(real_z_index)),
        'Gaussian-smoothed real': one_saved_prediction(rows, source='real_gaussian', run_name=run_name, dataset_size=int(dataset_size), transform=f'gaussian_smoothing__{suffix}', z_index=int(real_z_index)),
        'generated': one_saved_prediction(rows, source='generated', run_name=run_name, dataset_size=int(dataset_size), transform=f'generated__{suffix}', z_index=int(generated_sample_index)),
    }
    truth_values = np.asarray([float(record['theta_true']) for record in records.values()])
    if not np.allclose(truth_values, truth_values[0], rtol=0.0, atol=1.0e-7):
        raise ValueError(f'C4 saved rows disagree on truth values: {truth_values.tolist()}')
    return records

def apply_saved_transform_for_display(images, transform_name, *, k_bins=None, transfer_values=None, sigma=None):
    array = np.asarray(images, dtype=np.float32)
    if transform_name == 'power-matched real':
        transformed, _ = transfer_transform(np.asarray(k_bins), np.asarray(transfer_values))(array)
    elif transform_name == 'Gaussian-smoothed real':
        transformed, _ = gaussian_smoothing_transform(float(sigma))(array)
    elif (composite := re.fullmatch(r'dihedral_g([0-7])__roll_dx(-?\d+)_dy(-?\d+)', transform_name)):
        oriented, _ = get_transform(f'dihedral_g{composite.group(1)}')(array)
        transformed, _ = get_transform(f'roll_dx{composite.group(2)}_dy{composite.group(3)}')(oriented)
    else:
        transformed, _ = get_transform(transform_name)(array)
    return transformed

def load_real_display_image(sim_index, z_index):
    with np.load(ENCODER_PATH, allow_pickle=True) as payload:
        normalization = payload['normalization'].item()
    images, theta, simulations, slices = load_heldout_real_slices(
        DATA_ROOT,
        np.asarray([sim_index], dtype=np.int64),
        slices_per_sim=128,
        norm=normalization,
    )
    match = (simulations == int(sim_index)) & (slices == int(z_index))
    if int(match.sum()) != 1:
        raise ValueError(f'Could not locate real sim={sim_index}, z={z_index}')
    return images[match][0:1], theta[match][0]

def shared_image_limits(images):
    pixels = np.concatenate([np.asarray(image).ravel() for image in images])
    low, high = np.quantile(pixels, [0.01, 0.99])
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low, high = float(np.min(pixels)), float(np.max(pixels) + 1.0e-6)
    return float(low), float(high)

def image_plane(image):
    array = np.asarray(image)
    while array.ndim > 2:
        array = array[0]
    return array

def prediction_title(label, theta_true, theta_pred, baseline_pred):
    delta = float(theta_pred) - float(baseline_pred)
    return f'{label}\ntrue Omega_m={theta_true:.4f}\npredicted={theta_pred:.4f}; change={delta:+.4f}'

def load_generated_display_image(sample_path, sim_index, sample_index):
    with np.load(sample_path, allow_pickle=True) as payload:
        samples = np.asarray(payload['samples'], dtype=np.float32)
        heldout = np.asarray(payload['heldout_indices'], dtype=np.int64)
        count = int(payload['samples_per_cosmology'])
    positions = np.flatnonzero(heldout == int(sim_index))
    if len(positions) != 1 or not 0 <= int(sample_index) < count:
        raise ValueError(f'Generated sample lookup failed for sim={sim_index}, sample={sample_index}')
    return samples[int(positions[0]) * count + int(sample_index) : int(positions[0]) * count + int(sample_index) + 1]

### C0 — Does moving the same pattern change predicted $\Omega_m$?

These four panels contain exactly the same physical values. Rotation, reflection, and periodic shifting only change where the pattern appears in the square. A stable predictor should not depend strongly on these presentation choices.

In [ ]:
omega_original_rows = read_saved_prediction_rows(
    C4_CSV,
    {'parameter': 'Omega_m', 'source': 'real_original'},
)
EXAMPLE = select_representative_example(omega_original_rows)
EXAMPLE_SIM = EXAMPLE['sim_index']
EXAMPLE_Z = EXAMPLE['z_index']
REAL_EXAMPLE, REAL_EXAMPLE_THETA = load_real_display_image(EXAMPLE_SIM, EXAMPLE_Z)
if not np.isclose(float(REAL_EXAMPLE_THETA[0]), EXAMPLE['theta_true'], rtol=0.0, atol=1.0e-7):
    raise ValueError('Raw real-map truth does not match the saved original prediction row')

transform_records = transform_manifest['transforms']
rotation_name = next(record['name'] for record in transform_records if record.get('family') == 'dihedral' and int(record.get('dihedral_g')) == 1 and int(record.get('roll_dx') or 0) == 0 and int(record.get('roll_dy') or 0) == 0)
reflection_name = next(record['name'] for record in transform_records if record.get('family') == 'dihedral' and int(record.get('dihedral_g')) == 4 and int(record.get('roll_dx') or 0) == 0 and int(record.get('roll_dy') or 0) == 0)
shift_name = sorted(record['name'] for record in transform_records if record.get('family') == 'roll' and int(record.get('dihedral_g')) == 0)[0]
C0_TRANSFORM_NAMES = ('identity', rotation_name, reflection_name, shift_name)
C0_DISPLAY_LABELS = ('Original', 'Rotated 90 degrees', 'Reflected', 'Periodically shifted')

c0_saved = read_saved_prediction_rows(
    TRANSFORM_CSV,
    {'parameter': 'Omega_m', 'sim_index': EXAMPLE_SIM, 'z_index': EXAMPLE_Z, 'transform': C0_TRANSFORM_NAMES},
)
c0_records = saved_transform_records(c0_saved, C0_TRANSFORM_NAMES)
c0_predictions = {name: float(record['theta_pred']) for name, record in c0_records.items()}
c0_images = [
    apply_saved_transform_for_display(REAL_EXAMPLE, name)
    for name in C0_TRANSFORM_NAMES
]
c0_baseline = c0_predictions['identity']
c0_vmin, c0_vmax = shared_image_limits(c0_images)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
for axis, label, name, image in zip(axes, C0_DISPLAY_LABELS, C0_TRANSFORM_NAMES, c0_images):
    c0_artist = axis.imshow(image_plane(image), cmap='viridis', vmin=c0_vmin, vmax=c0_vmax, interpolation='nearest')
    axis.set_title(prediction_title(label, float(c0_records[name]['theta_true']), c0_predictions[name], c0_baseline), fontsize=9)
    axis.set_xticks(()); axis.set_yticks(())
fig.colorbar(c0_artist, ax=axes.tolist(), shrink=0.72, label='normalized map value')
fig.suptitle('C0: same map values, different orientation or location')
plt.show()

c0_changes = np.asarray([c0_predictions[name] - c0_baseline for name in C0_TRANSFORM_NAMES[1:]])
display(Markdown(
    f'**Did orientation/location change predicted Omega_m?**  '
    f'For deterministic sim `{EXAMPLE_SIM}`, slice `{EXAMPLE_Z}`, the largest absolute change among the displayed controls is '
    f'`{np.max(np.abs(c0_changes)):.5f}`. This is a descriptive change, not a pass/fail threshold.'
))
print('Deterministic example: sim_index =', EXAMPLE_SIM, '; z_index =', EXAMPLE_Z, '; selection used true Omega_m only')

### C1 — Which structure sizes matter for predicted $\Omega_m$?

- **Low-pass** keeps broad, slowly varying structure and removes fine detail.
- **High-pass** keeps fine detail and removes broad structure.

Each row uses one spatial cutoff. The original, low-pass, and high-pass panels share a color scale within that row. “Fourier transform” (FFT in the technical section) is simply the calculation used to separate broad from fine spatial patterns.

In [ ]:
C1_DISPLAY_CUTS = (8.0, 24.0, 40.0)
c1_names = ['identity']
for cutoff in C1_DISPLAY_CUTS:
    label = f'{cutoff:g}'
    c1_names.extend([f'lowpass_kcut{label}_sharp', f'highpass_kcut{label}_sharp'])
c1_saved = read_saved_prediction_rows(
    TRANSFORM_CSV,
    {'parameter': 'Omega_m', 'sim_index': EXAMPLE_SIM, 'z_index': EXAMPLE_Z, 'transform': c1_names},
)
c1_records = saved_transform_records(c1_saved, c1_names)
c1_predictions = {name: float(record['theta_pred']) for name, record in c1_records.items()}
c1_baseline = c1_predictions['identity']

fig, axes = plt.subplots(len(C1_DISPLAY_CUTS), 3, figsize=(12, 11), constrained_layout=True)
c1_observations = []
for row_index, cutoff in enumerate(C1_DISPLAY_CUTS):
    label = f'{cutoff:g}'
    low_name = f'lowpass_kcut{label}_sharp'
    high_name = f'highpass_kcut{label}_sharp'
    row_names = ('identity', low_name, high_name)
    row_labels = ('Original', f'Low-pass, cutoff {label}\n(broad structure remains)', f'High-pass, cutoff {label}\n(fine structure remains)')
    row_images = [apply_saved_transform_for_display(REAL_EXAMPLE, name) for name in row_names]
    vmin, vmax = shared_image_limits(row_images)
    for axis, panel_label, name, image in zip(axes[row_index], row_labels, row_names, row_images):
        c1_artist = axis.imshow(image_plane(image), cmap='viridis', vmin=vmin, vmax=vmax, interpolation='nearest')
        axis.set_title(prediction_title(panel_label, float(c1_records[name]['theta_true']), c1_predictions[name], c1_baseline), fontsize=8)
        axis.set_xticks(()); axis.set_yticks(())
    fig.colorbar(c1_artist, ax=axes[row_index].tolist(), shrink=0.68, label='normalized map value')
    c1_observations.append({
        'cutoff': cutoff,
        'low_pass_change': c1_predictions[low_name] - c1_baseline,
        'high_pass_change': c1_predictions[high_name] - c1_baseline,
    })
fig.suptitle('C1: original map versus broad-only and fine-only information')
plt.show()

c1_observed = pd.DataFrame(c1_observations)
fig, axis = plt.subplots(figsize=(7, 4), constrained_layout=True)
axis.axhline(0, color='0.25', linestyle=':', linewidth=1)
axis.plot(c1_observed['cutoff'], c1_observed['low_pass_change'], color='tab:blue', marker='o', label='broad structure remains')
axis.plot(c1_observed['cutoff'], c1_observed['high_pass_change'], color='tab:orange', marker='s', linestyle='--', label='fine structure remains')
axis.set_xlabel('spatial cutoff'); axis.set_ylabel('change in predicted Omega_m from original')
axis.set_title('C1 saved prediction changes for the displayed map')
axis.legend(); axis.grid(alpha=0.2)
plt.show()
display(Markdown(
    f'**What happens when broad or fine structure is removed?**  '
    f'Across the displayed cutoffs, keeping only broad structure changes predicted Omega_m by at most '
    f'`{c1_observed.low_pass_change.abs().max():.5f}`; keeping only fine structure changes it by at most '
    f'`{c1_observed.high_pass_change.abs().max():.5f}`. The curves below show the full held-out population.'
))

### C4 — Is the generated-map shift explained only by missing spatial power?

For each generated model, the same original real map is altered in two controlled ways and compared with one generated map:

1. **Power-matched real:** uses the measured generated/real power difference at each spatial scale.
2. **Gaussian-smoothed real:** uses a simple smooth blur fitted to that power difference.
3. **Generated:** uses the same cosmology, different realization. It is **not** a pixelwise “after” image of the real panel.

Every displayed prediction still comes from the SAME frozen predictor and the completed saved CSV.

In [ ]:
power_for_gallery = strict_json(C4_DIR / 'power_transfer_curves.json')
c4_example_rows = read_saved_prediction_rows(
    C4_CSV,
    {'parameter': 'Omega_m', 'sim_index': EXAMPLE_SIM},
)
c4_original = one_saved_prediction(
    c4_example_rows,
    source='real_original', run_name='real_original', transform='identity', z_index=EXAMPLE_Z,
)
c4_original_prediction = float(c4_original['theta_pred'])
c4_gallery_summary = []

fig, axes = plt.subplots(len(EXPECTED_RUNS), 4, figsize=(17, 8), constrained_layout=True)
for row_index, run_name in enumerate(sorted(EXPECTED_RUNS)):
    run_power = power_for_gallery['runs'][run_name]
    dataset_size = int(run_power['dataset_size'])
    run_rows = c4_example_rows[c4_example_rows['run_name'].astype(str) == run_name].copy()
    generated_rows = run_rows[run_rows['source'] == 'generated'].copy()
    generated_sample = int(generated_rows['z_index'].astype(int).min())
    c4_records = c4_saved_records_for_display(
        c4_example_rows,
        run_name=run_name,
        dataset_size=dataset_size,
        real_z_index=EXAMPLE_Z,
        generated_sample_index=generated_sample,
    )

    measured_image = apply_saved_transform_for_display(
        REAL_EXAMPLE,
        'power-matched real',
        k_bins=run_power['k_bins'],
        transfer_values=run_power['measured_transfer'],
    )
    gaussian_image = apply_saved_transform_for_display(
        REAL_EXAMPLE,
        'Gaussian-smoothed real',
        sigma=run_power['gaussian_sigma_pixels'],
    )
    generated_image = load_generated_display_image(
        Path(run_power['sample_path']), EXAMPLE_SIM, generated_sample,
    )
    row_images = [REAL_EXAMPLE, measured_image, gaussian_image, generated_image]
    row_predictions = [
        float(c4_records[label]['theta_pred']) for label in C4_GALLERY_COLUMNS
    ]
    row_truths = [float(c4_records[label]['theta_true']) for label in C4_GALLERY_COLUMNS]
    vmin, vmax = shared_image_limits(row_images)
    run_label = RUN_LABELS[run_name]
    for axis, panel_label, image, truth, prediction in zip(axes[row_index], C4_GALLERY_COLUMNS, row_images, row_truths, row_predictions):
        c4_artist = axis.imshow(image_plane(image), cmap='viridis', vmin=vmin, vmax=vmax, interpolation='nearest')
        axis.set_title(prediction_title(panel_label, truth, prediction, c4_original_prediction), fontsize=8)
        axis.set_xticks(()); axis.set_yticks(())
    fig.colorbar(c4_artist, ax=axes[row_index].tolist(), shrink=0.68, label='normalized map value')
    axes[row_index, 0].set_ylabel(run_label)

    generated_change = row_predictions[3] - c4_original_prediction
    measured_change = row_predictions[1] - c4_original_prediction
    gaussian_change = row_predictions[2] - c4_original_prediction
    c4_gallery_summary.append({
        'run_name': run_name,
        'generated_sample_index': generated_sample,
        'generated_change': generated_change,
        'power_matched_real_change': measured_change,
        'Gaussian_smoothed_real_change': gaussian_change,
        'remaining_difference_power_matched_vs_generated': abs(measured_change - generated_change),
    })
    print('C4 identifiers:', run_name, 'sim_index =', EXAMPLE_SIM, 'real z_index =', EXAMPLE_Z, 'generated sample_index =', generated_sample)
fig.suptitle('C4: original real, controlled real variants, and a same-cosmology generated realization')
plt.show()

c4_gallery_summary = pd.DataFrame(c4_gallery_summary)
display(Markdown('**Does matching the power deficit reproduce the generated-map Omega_m shift?**'))
display(c4_gallery_summary)
display(Markdown(
    'Read each row numerically: compare `power_matched_real_change` with `generated_change`. '
    'The remaining absolute difference is reported without inventing a pass/fail threshold. '
    'This test addresses only the measured two-point power deficit; it does not match the one-point distribution or higher-order structure.'
))

## Technical details

The visual walkthrough above follows one deterministic example. The following population-level checks use all held-out rows and all six parameters.

- **RMSE (root mean squared error):** typical prediction error magnitude; lower is better.
- **Bias:** average signed prediction error; zero means no average offset.
- **Slope:** response of predicted versus true values; one is the ideal reference.
- **FFT (fast Fourier transform):** the calculation used to separate spatial scales.

Error bars are the stored uncertainty intervals. These plots are diagnostic evidence, not automatic scientific pass/fail tests.

### C0 population check — orientation and location stability

These ratios compare prediction spread across rotated/reflected/shifted views with ordinary slice-to-slice spread. The worst cases table is descriptive, not a binary acceptance test.

In [ ]:
c0_summary = strict_json(TRANSFORM_DIR / 'c0_symmetry_summary.json')
family_summary = pd.DataFrame.from_dict(c0_summary['family_summary'], orient='index').reset_index(names='family')
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
x = np.arange(len(family_summary))
for axis, value, low, high, title in [(axes[0], 'median_std_ratio', 'median_std_ratio_ci_low', 'median_std_ratio_ci_high', 'C0 transform spread / identity spread'), (axes[1], 'median_range_ratio', 'median_range_ratio_ci_low', 'median_range_ratio_ci_high', 'C0 transform range / identity range')]:
    y = family_summary[value].to_numpy(float)
    lower = y - family_summary[low].to_numpy(float)
    upper = family_summary[high].to_numpy(float) - y
    axis.errorbar(x, y, yerr=[lower, upper], fmt='o', color='tab:blue', capsize=4)
    axis.set_xticks(x, family_summary['family'])
    axis.set_title(title)
    axis.set_ylabel('ratio (descriptive)')
    axis.grid(alpha=0.25)
fig.suptitle('C0 symmetry and translation stability')
plt.show()

worst = pd.DataFrame(c0_summary['per_slice']).sort_values('std_over_within_sim_std', ascending=False).head(10)
display(worst[['family', 'sim_index', 'z_index', 'std_over_within_sim_std', 'range_over_within_sim_range']])

### C1 population check — spatial cutoffs

The full curves show all six parameters. Solid/dashed lines distinguish sharp and tapered cutoffs; circles/squares distinguish broad-only and fine-only inputs. The Fourier round-trip reference checks that converting to and from Fourier space alone does not change the result.

In [ ]:
c1_summary = strict_json(TRANSFORM_DIR / 'c1_scale_cut_summary.json')
c1 = pd.DataFrame(c1_summary['curves'])
c1 = c1[c1['grain'] == 'per_cosmology'].copy()
def assert_unique_grain(frame, columns, label):
    duplicate = frame.duplicated(columns, keep=False)
    if duplicate.any():
        sample = frame.loc[duplicate, columns].head(3).to_dict('records')
        raise ValueError(f'{label} has duplicate analytical grains: {sample}')

assert_unique_grain(c1, ['parameter', 'transform', 'transform_family', 'window', 'k_cut'], 'C1 summary')
parameter_facets = list(PARAMETER_ORDER)
if set(parameter_facets) != set(c1['parameter'].astype(str)):
    raise ValueError(f'C1 parameter coverage is incomplete: {parameter_facets}')
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False, constrained_layout=True)
styles = {'sharp': '-', 'hann': '--'}
markers = {'lowpass': 'o', 'highpass': 's'}
for axis, parameter in zip(axes.ravel(), parameter_facets):
    subset = c1[c1['parameter'].astype(str) == parameter]
    for (family, window), group in subset[subset['transform_family'].isin(['lowpass', 'highpass'])].groupby(['transform_family', 'window']):
        group = group.sort_values('k_cut')
        y = group['rmse'].to_numpy(float)
        yerr = [y - group['rmse_ci_low'].to_numpy(float), group['rmse_ci_high'].to_numpy(float) - y]
        axis.errorbar(group['k_cut'], y, yerr=yerr, color='tab:blue' if family == 'lowpass' else 'tab:orange', linestyle=styles[window], marker=markers[family], capsize=2, label=f'{family}, {window}')
    identity = subset[subset['transform'] == 'identity']
    null = subset[subset['transform_family'] == 'fft_roundtrip_null']
    if len(identity) != 1 or len(null) != 1:
        raise ValueError(f'C1 parameter-specific identity/null reference missing for {parameter}')
    axis.axhline(float(identity['rmse'].iloc[0]), color='0.25', linestyle='-.', label='identity')
    axis.axhline(float(null['rmse'].iloc[0]), color='0.45', linestyle=':', label='FFT round-trip null')
    axis.set_title(parameter)
    axis.set_xlabel('k cut (ordered)')
    axis.set_ylabel('RMSE (descriptive)')
    axis.grid(alpha=0.25)
axes[0, 0].legend(fontsize=8)
fig.suptitle('C1 ordered k-cut probe RMSE by parameter; lines show stored uncertainty')
plt.show()
compact_c1 = c1[['parameter', 'transform_family', 'window', 'k_cut', 'rmse', 'rmse_ci_low', 'rmse_ci_high', 'bias', 'bias_ci_low', 'bias_ci_high', 'slope', 'slope_ci_low', 'slope_ci_high', 'out_of_range_fraction']]
display(compact_c1.sort_values(['parameter', 'transform_family', 'window', 'k_cut']).head(24))

### C4 population check — original, controlled real, and generated maps

These panels compare population-level RMSE, bias, slope, power, and one-point distributions for both generated runs. Remember: matching two-point power does not match the one-point distribution or higher-order structure.

In [ ]:
c4_metrics = strict_json(C4_DIR / 'probe_degradation_metrics.json')
power = strict_json(C4_DIR / 'power_transfer_curves.json')
histograms = strict_json(C4_DIR / 'field_histograms.json')
if c4_metrics.get('limitation') != C4_LIMITATION or power.get('limitation') != C4_LIMITATION:
    raise ValueError('C4 limitation text is missing or changed')
metrics = pd.DataFrame(c4_metrics['metrics'])
metrics = metrics[metrics['grain'] == 'per_cosmology'].copy()
assert_unique_grain(metrics, ['parameter', 'grain', 'source', 'run_name', 'transform'], 'C4 metrics')
source_order = ['real_original', 'real_measured_transfer', 'real_gaussian', 'generated']
if set(metrics['source'].astype(str)) != set(source_order):
    raise ValueError('C4 source coverage is incomplete')
baseline = metrics[metrics['source'] == 'real_original'].copy()
if len(baseline) == 0 or set(baseline['parameter']) != PARAMETERS:
    raise ValueError('C4 real_original baseline is incomplete')
baseline_rows = []
for run_name in sorted(EXPECTED_RUNS):
    repeated = baseline.copy()
    repeated['run_name'] = run_name
    repeated['transform'] = 'identity__baseline__' + run_name
    baseline_rows.append(repeated)
comparison = pd.concat([metrics[metrics['source'] != 'real_original']] + baseline_rows, ignore_index=True)
assert_unique_grain(comparison, ['parameter', 'grain', 'source', 'run_name', 'transform'], 'C4 comparison')
if set(comparison['run_name'].astype(str)) != EXPECTED_RUNS:
    raise ValueError('C4 comparison does not cover both runs')
metrics_to_plot = [('rmse', 'RMSE'), ('bias', 'Bias'), ('slope', 'Slope')]
source_short_labels = {'real_original': 'original real', 'real_measured_transfer': 'measured transfer', 'real_gaussian': 'Gaussian', 'generated': 'generated'}
source_styles = {'real_original': ('0.25', 'o', '-'), 'real_measured_transfer': ('tab:blue', 's', '--'), 'real_gaussian': ('tab:orange', '^', ':'), 'generated': ('0.10', 'D', '-.')}
run_names = sorted(EXPECTED_RUNS)
if set(run_names) != set(comparison['run_name'].astype(str)):
    raise ValueError('C4 comparison run set is not exactly the expected run set')
RUN_LABELS = {'nf_cond_bias_hi_u128_d2p07_n128_200k': 'N=128, d=2.07', 'nf_cond_bias_hi_u128_d2p14_n16384_200k': 'N=16,384, d=2.14'}
if set(RUN_LABELS) != set(run_names):
    raise ValueError('C4 run labels do not cover the expected runs')
fig, axes = plt.subplots(2, 3, figsize=(17, 9), squeeze=False, constrained_layout=True)
for row_index, run_name in enumerate(run_names):
    run_label = RUN_LABELS[run_name]
    run_rows = comparison[(comparison['run_name'] == run_name) & (comparison['parameter'] == FOCUS_PARAMETER)]
    if set(run_rows['source']) != set(source_order):
        raise ValueError(f'C4 source comparison is incomplete for {run_name}')
    run_rows = run_rows.set_index('source').loc[source_order].reset_index()
    x = np.arange(len(source_order))
    for column_index, (metric, label) in enumerate(metrics_to_plot):
        axis = axes[row_index, column_index]
        y = run_rows[metric].to_numpy(float)
        lower = y - run_rows[f'{metric}_ci_low'].to_numpy(float)
        upper = run_rows[f'{metric}_ci_high'].to_numpy(float) - y
        for index, row in run_rows.iterrows():
            color, marker, _ = source_styles[row['source']]
            axis.errorbar(x[index], y[index], yerr=[[lower[index]], [upper[index]]], color=color, marker=marker, linestyle='none', capsize=3, label=source_short_labels[row['source']])
        if metric == 'bias':
            axis.axhline(0, color='0.45', linestyle=':', linewidth=1, label='zero reference')
        elif metric == 'slope':
            axis.axhline(1, color='0.45', linestyle=':', linewidth=1, label='unit reference')
        axis.set_xticks(x, [source_short_labels[source] for source in source_order], rotation=28, ha='right')
        axis.set_title(f'Run {row_index + 1}: {run_label} — {label}')
        axis.set_ylabel(label + ' (descriptive)')
        axis.grid(axis='y', alpha=0.25)
        if row_index == 0 and column_index == 2:
            axis.legend(fontsize=8, loc='best')
fig.suptitle(f'C4 grouped probe metrics for {FOCUS_PARAMETER}; stored CIs')
plt.show()
display(comparison[comparison['parameter'] == FOCUS_PARAMETER][['run_name', 'source', 'rmse', 'rmse_ci_low', 'rmse_ci_high', 'bias', 'bias_ci_low', 'bias_ci_high', 'slope', 'slope_ci_low', 'slope_ci_high']].sort_values(['run_name', 'source']))

RUN_COLORS = {run_names[0]: 'tab:blue', run_names[1]: 'tab:orange'}
fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
for run_name in run_names:
    run = power['runs'][run_name]
    color = RUN_COLORS[run_name]
    axes[0].plot(run['k_bins'], run['power_ratio'], color=color, marker='o', linestyle='-', label=f'{run_name}: generated / real')
    axes[1].plot(run['k_bins'], run['measured_transfer'], color=color, marker='s', linestyle='--', label=f'{run_name}: measured transfer')
    axes[1].plot(run['k_bins'], run['gaussian_transfer'], color=color, marker='^', linestyle=':', label=f'{run_name}: Gaussian transfer')
axes[0].set_title('C4 generated / real power ratio')
axes[0].set_xlabel('k bin'); axes[0].set_ylabel('power ratio')
axes[1].set_title('C4 measured and Gaussian transfer')
axes[1].set_xlabel('k bin'); axes[1].set_ylabel('transfer amplitude')
for axis in axes: axis.legend(fontsize=8); axis.grid(alpha=0.25)
plt.show()

fig, axes = plt.subplots(1, len(EXPECTED_RUNS), figsize=(15, 4), squeeze=False, constrained_layout=True)
hist_baseline = histograms['real_original']
for axis, run_name in zip(axes.ravel(), sorted(EXPECTED_RUNS)):
    run = histograms['runs'][run_name]
    series = [('real_original', hist_baseline), ('real_measured_transfer', run['real_measured_transfer']), ('real_gaussian', run['real_gaussian']), ('generated', run['generated'])]
    for source, values in series:
        color, _, linestyle = source_styles[source]
        edges = np.asarray(values['bin_edges'], dtype=float)
        heights = np.asarray(values['hist'], dtype=float)
        if len(edges) != len(heights) + 1:
            raise ValueError(f'Histogram bin schema is invalid for {run_name}: {source}')
        centers = 0.5 * (edges[:-1] + edges[1:])
        axis.hist(centers, bins=edges, weights=heights, histtype='step', color=color, linestyle=linestyle, linewidth=1.5, label=source_short_labels[source])
    axis.set_title(f'C4 one-point field PDF: {run_name}')
    axis.set_xlabel('field value'); axis.set_ylabel('density')
    axis.legend(fontsize=7); axis.grid(alpha=0.2)
plt.show()

## Takeaways

Use the image galleries first: they show exactly what changed and the saved $\Omega_m$ prediction for the same deterministic example. Then use the population plots to check whether that example reflects the broader held-out set.

The status below refers only to data/provenance quality. It does not declare that a scientific control “passed.” The C4 conclusion is deliberately narrow: it tests whether the measured two-point power deficit alone can account for the prediction shift.

In [ ]:
STATUS_LABELS = ('Ready to share', 'Share with caveats', 'Needs revision')
structural_status = 'verified'
status = 'Share with caveats' if structural_status == 'verified' else 'Needs revision'
c0_observed = family_summary[['family', 'median_std_ratio', 'median_std_ratio_ci_low', 'median_std_ratio_ci_high', 'median_range_ratio', 'median_range_ratio_ci_low', 'median_range_ratio_ci_high']].copy()
closest_rows = []
for run_name in sorted(EXPECTED_RUNS):
    run_rows = comparison[(comparison['run_name'] == run_name) & (comparison['parameter'] == FOCUS_PARAMETER)].copy()
    generated_value = float(run_rows.loc[run_rows['source'] == 'generated', 'rmse'].iloc[0])
    alternatives = run_rows[run_rows['source'] != 'generated'].copy()
    alternatives['absolute_rmse_difference_from_generated'] = (alternatives['rmse'] - generated_value).abs()
    closest = alternatives.sort_values(['absolute_rmse_difference_from_generated', 'source']).iloc[0]
    closest_rows.append({'run_name': run_name, 'closest_non_generated_source_by_rmse': closest['source'], 'generated_rmse': generated_value, 'closest_rmse': float(closest['rmse']), 'absolute_rmse_difference_from_generated': float(closest['absolute_rmse_difference_from_generated'])})
display(Markdown(f'## tl;dr\n\n**Status: {status}** — structural status: **{structural_status}**. The observations below are exact values from the saved summaries; no scientific pass threshold is applied. C4 remains subject to the limitation that matching two-point power does not match the one-point PDF or higher-order structure.'))
display(Markdown('**Observed C0 family ratios**'))
display(c0_observed)
display(Markdown(f'**Observed C4 closest non-generated source by {FOCUS_PARAMETER} RMSE**'))
display(pd.DataFrame(closest_rows))